[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/08-ai-engineering/03_context-memory.ipynb)

*AI Engineering · Notebook 3 of 10*

# Context Engineering & Agent Memory

Decide what the model sees: token budgets, trimming and compaction, ordering for prompt caching, 'lost in the middle', clearing old tool results, just-in-time retrieval, and long-term memory — extracting facts, add/update/skip decisions, recall with recency, and resolving conflicts.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** The context window is a small desk: put only what the task needs on it, keep the important papers at the edges, and file the rest in memory.

**What interviewers really test here:** Trimming vs compaction, prompt caching and prefix order, lost in the middle, and add/update/skip memory with recency.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. From prompt engineering to context engineering *(Basics)*
- [ ] 2. Keeping long conversations inside the window *(Core)*
- [ ] 3. Order matters: caching and 'lost in the middle' *(Core)*
- [ ] 4. Choosing what to include *(Core)*
- [ ] 5. Memory that lasts beyond one conversation *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [ ]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
import numpy as np
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### What's in the window

| Part | Changes | Typical lever |
| --- | --- | --- |
| System prompt | rarely | keep it clear and short; cache it |
| Tool definitions | rarely | fewer, well-described tools; cache them |
| Documents / retrieval | per request | retrieve less but better; order by importance |
| Memory | per user | recall only the top few facts |
| History | every turn | trim, clear tool results, compact |
| Question | every turn | put it last |

### Kinds of memory

- **Working memory**: the context window itself.
- **Short-term**: recent turns of this conversation.
- **Long-term semantic**: stable facts and preferences ("prefers email").
- **Episodic**: past events and conversations, retrievable later.
- **Procedural**: learned instructions and notes about how to do tasks.

---
# Part 2: Lessons, step by step

## Lesson 1: From prompt engineering to context engineering

*Level: Basics*

A model only knows what is in its **context window** right now: the system prompt, tool definitions, retrieved documents, the conversation so far, tool results and the question. **Context engineering** is deciding what goes in, in what order, and what comes out as the task goes on.



More isn't better. Every token costs money and time, and models get worse at using information as the context fills up ("context rot"): details in the middle of a long context are easily missed. The goal is **the smallest set of high-signal tokens** that lets the model succeed.



Start by measuring. A rough estimate is 4 characters per English token; for real numbers use the provider's token-counting endpoint (Claude's API has one) and look at the `usage` field of every response.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 2: Keeping long conversations inside the window

*Level: Core*

Three tools, from simplest to smartest:




1. **Trim**: keep the system prompt and the most recent turns; drop the oldest. Cheap, but the agent forgets early decisions.
2. **Clear tool results**: after the agent has used a big tool output (a file, a web page), replace it with a short placeholder. Usually the biggest win for agents. Some APIs, including Claude's, can do this automatically ("context editing").
3. **Compact**: when the history passes a threshold, ask the model to summarise the oldest part (decisions made, open questions, file names) and continue from the summary. This is how long-running coding agents keep going for hours.



Remember that every turn re-sends the whole history: without trimming, the total input over a conversation grows with the **square** of its length.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [ ]:
import matplotlib.pyplot as plt


def growth_chart():
    turns = list(range(1, 41))
    plain, hist, compacted, total = [], 0, [], 0
    run = 0
    for t in turns:
        run += 2000 * t
        plain.append(run)
        hist = hist + 2000 if hist + 2000 <= 30000 else 5000
        total += hist
        compacted.append(total)
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.plot(turns, plain, label="re-send everything")
    ax.plot(turns, compacted, label="compact at 30k")
    ax.set_xlabel("turn"); ax.set_ylabel("total input tokens"); ax.legend()


growth_chart()

**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q3](#q3), [Q10](#q10), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 3: Order matters: caching and 'lost in the middle'

*Level: Core*

**Prompt caching** lets the provider reuse the processed form of a prompt *prefix* it has seen recently. Cached input is billed at a small fraction of the normal price (on Claude, reads cost about 10% and writing the cache about 125%, with a minimum cacheable length), and it's faster. A cache hit needs the start of the prompt to be **byte-for-byte identical**, so put stable content first: system prompt, tool definitions, long documents — then per-user data — then the changing question.



```python
client.messages.create(model="claude-opus-5-5", max_tokens=1000,
    system=[{"type": "text", "text": LONG_POLICY_DOC, "cache_control": {"type": "ephemeral"}}],
    messages=[{"role": "user", "content": question}])
```



Within the content, put the most important documents at the **start and end**, and restate the task at the end of a long prompt.

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q12](#q12), [Q20](#q20), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 4: Choosing what to include

*Level: Core*

- **Budget by section**: give each part (system, memory, documents, history) a minimum and a share of what's left, so one huge section can't crowd out the rest.
- **Just-in-time retrieval**: instead of preloading every file, give the agent the list of names and tools to read what it needs — the way a person uses a file browser.
- **Dynamic few-shot examples**: pick the examples most similar to the current input instead of always the same ones.

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q4](#q4), [Q14](#q14), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 5: Memory that lasts beyond one conversation

*Level: Advanced*

Long-term memory is context engineering across sessions. A typical pipeline:




1. **Extract** candidate facts from the conversation (usually with an LLM: "list stable facts about the user").
2. **Consolidate** against the store: **ADD** new facts, **UPDATE** changed ones (keeping history), **SKIP** duplicates. Resolve conflicts by recency or by asking the user.
3. **Recall** before answering: score memories by relevance to the question (keywords, embeddings or both) times a **recency decay**, and include only the top few.



Tiers help: a short-term buffer of recent messages, and a long-term store for facts. For long tasks, agents also keep **notes files** (a to-do list, decisions made) that they re-read after compaction. Some APIs provide a memory tool so the model can create and edit such files itself.



Memory holds personal data: let users see and delete it, and never store secrets.

**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q17](#q17), [Q18](#q18), [Q7](#q7), [Q5](#q5), [Q6](#q6). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Estimate Tokens

🟢 Warm-up · *Budgets start with counting*

Estimate tokens as ⌈characters ÷ 4⌉ per message. Return `(tokens per message, total)`.

**Examples**

- `messages = ['You are a helpful assistant.', 'Summarise MCP in one line.', '']` → `([7, 7, 0], 14)`

In [ ]:
def count_tokens(messages):
    """
    Estimate tokens as ⌈characters ÷ 4⌉ per message. Return (tokens per message,
    total).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_tokens, [
    ((['You are a helpful assistant.', 'Summarise MCP in one line.', ''],), ([7, 7, 0], 14)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with ceiling division.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension with the `toks` helper, then `sum`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with ceiling division.

```python
def count_tokens(messages):
    per = []
    for m in messages:
        n = len(m) // 4
        if len(m) % 4:
            n += 1
        per.append(n)
    return (per, sum(per))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension with the `toks` helper, then `sum`.

```python
def toks(text):
    return (len(text) + 3) // 4


def count_tokens(messages):
    per = [toks(m) for m in messages]
    return (per, sum(per))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with ceiling division. That costs O(n) time and O(n) space."

"I can do better: A comprehension with the `toks` helper, then `sum`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/estimate-tokens)

<a id="q2"></a>
### Q2. Put Stable Content First

🟢 Warm-up · *Cache-friendly order*

Prompt caches reuse an identical **prefix**. Order blocks so everything that never changes comes first (keeping their original order), then things that change per user, then per-request content. Each block is `(name, 'static' | 'per_user' | 'per_request')`; return the names in the new order.

**Examples**

- `blocks = [('question', 'per_request'), ('system', 'static'), ('user_profile', 'per_user'), ('tools', 'static'), ('retrieved_docs', 'per_request')]` → `['system', 'tools', 'user_profile', 'question', 'retrieved_docs']`

In [ ]:
def cache_order(blocks):
    """
    Prompt caches reuse an identical prefix. Order blocks so everything that
    never changes comes first (keeping their original order), then things that
    change per user, then per-request content. Each block is (name, 'static' |
    'per_user' | 'per_request'); return the names in the new order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cache_order, [
    (([('question', 'per_request'), ('system', 'static'), ('user_profile', 'per_user'), ('tools', 'static'), ('retrieved_docs', 'per_request')],), ['system', 'tools', 'user_profile', 'question', 'retrieved_docs']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Three passes, one per kind.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A stable `sorted` with a rank key.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Three passes, one per kind.

```python
def cache_order(blocks):
    out = []
    for kind in ["static", "per_user", "per_request"]:
        for name, k in blocks:
            if k == kind:
                out.append(name)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A stable `sorted` with a rank key.

```python
def cache_order(blocks):
    rank = {"static": 0, "per_user": 1, "per_request": 2}
    return [name for name, k in sorted(blocks, key=lambda b: rank[b[1]])]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Three passes, one per kind. That costs O(n) time and O(n) space."

"I can do better: A stable `sorted` with a rank key. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/put-stable-content-first)

<a id="q3"></a>
### Q3. Clear Old Tool Results

🟢 Warm-up · *Context editing*

Agents pile up tool outputs the model no longer needs. Replace every tool result except the newest `keep` with a 10-token placeholder. Messages are `(kind, tokens)`. Return `(tokens before, tokens after)`.

**Examples**

- `messages = [('user', 50), ('tool', 4000), ('assistant', 200), ('tool', 6000), ('tool', 3000), ('assistant', 150)], keep = 1` → `(13400, 3420)`

In [ ]:
def clear_tools(messages, keep):
    """
    Agents pile up tool outputs the model no longer needs. Replace every tool
    result except the newest keep with a 10-token placeholder. Messages are
    (kind, tokens). Return (tokens before, tokens after).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(clear_tools, [
    (([('user', 50), ('tool', 4000), ('assistant', 200), ('tool', 6000), ('tool', 3000), ('assistant', 150)], 1), (13400, 3420)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find tool positions, then rebuild sizes in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Indexes of tool messages; a comprehension decides each size.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Find tool positions, then rebuild sizes in a loop.

```python
def clear_tools(messages, keep):
    tool_idx = []
    for i, (kind, _) in enumerate(messages):
        if kind == "tool":
            tool_idx.append(i)
    old = tool_idx[:len(tool_idx) - keep] if keep else tool_idx
    after = 0
    for i, (kind, n) in enumerate(messages):
        after += 10 if i in old else n
    return (sum(n for _, n in messages), after)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Indexes of tool messages; a comprehension decides each size.

```python
def clear_tools(messages, keep):
    tools = [i for i, (k, _) in enumerate(messages) if k == "tool"]
    old = set(tools[:-keep] if keep else tools)
    return (sum(n for _, n in messages), sum(10 if i in old else n for i, (_, n) in enumerate(messages)))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find tool positions, then rebuild sizes in a loop. That costs O(n) time and O(n) space."

"I can do better: Indexes of tool messages; a comprehension decides each size. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/clear-old-tool-results)

<a id="q4"></a>
### Q4. Load Files Just in Time

🟢 Warm-up · *Preload vs on-demand*

An agent can preload every file into context, or list file names (5 tokens each) and read only what it needs. Return `(preload tokens, just-in-time tokens)` for the files it actually `used`.

**Examples**

- `files = {'README.md': 1200, 'api.py': 5400, 'models.py': 8000, 'tests.py': 6600, 'CHANGELOG.md': 9000}, used = ['README.md', 'api.py']` → `(30200, 6625)`

In [ ]:
def jit_tokens(files, used):
    """
    An agent can preload every file into context, or list file names (5 tokens
    each) and read only what it needs. Return (preload tokens, just-in-time
    tokens) for the files it actually used.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(jit_tokens, [
    (({'README.md': 1200, 'api.py': 5400, 'models.py': 8000, 'tests.py': 6600, 'CHANGELOG.md': 9000}, ['README.md', 'api.py']), (30200, 6625)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two running totals in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two `sum` expressions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two running totals in a loop.

```python
def jit_tokens(files, used):
    preload, jit = 0, 0
    for name, size in files.items():
        preload += size
        jit += 5
        if name in used:
            jit += size
    return (preload, jit)
```

**Cost:** O(f) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two `sum` expressions.

```python
def jit_tokens(files, used):
    return (sum(files.values()), 5 * len(files) + sum(files[u] for u in used))
```

**Cost:** O(f) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two running totals in a loop. That costs O(f) time and O(1) space."

"I can do better: Two `sum` expressions. That's O(f) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/load-files-just-in-time)

<a id="q5"></a>
### Q5. Short-Term Buffer, Long-Term Store

🟢 Warm-up · *Memory tiers*

Keep the last `n` messages in a short-term buffer. When a new message pushes one out, the oldest moves to long-term memory. Return `(short-term, long-term)` after all messages.

**Examples**

- `messages = ['m1', 'm2', 'm3', 'm4', 'm5'], n = 3` → `(['m3', 'm4', 'm5'], ['m1', 'm2'])`

In [ ]:
from collections import deque

def tiers(messages, n):
    """
    Keep the last n messages in a short-term buffer. When a new message pushes
    one out, the oldest moves to long-term memory. Return (short-term, long-
    term) after all messages.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tiers, [
    ((['m1', 'm2', 'm3', 'm4', 'm5'], 3), (['m3', 'm4', 'm5'], ['m1', 'm2'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list with pop(0).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `deque(maxlen=n)`: check whether it's full before appending.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list with pop(0).

```python
def tiers(messages, n):
    short, long_term = [], []
    for m in messages:
        short.append(m)
        if len(short) > n:
            long_term.append(short.pop(0))
    return (short, long_term)
```

**Cost:** O(n · b) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A `deque(maxlen=n)`: check whether it's full before appending.

```python
def tiers(messages, n):
    short, long_term = deque(maxlen=n), []
    for m in messages:
        if len(short) == n:
            long_term.append(short[0])
        short.append(m)
    return (list(short), long_term)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list with pop(0). That costs O(n · b) time and O(n) space."

"I can do better: A `deque(maxlen=n)`: check whether it's full before appending. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/short-term-buffer-long-term-store)

<a id="q6"></a>
### Q6. A Notes File the Agent Keeps

🟢 Warm-up · *Structured note-taking*

Long tasks outlive the context window, so agents keep a to-do file. Apply commands `('add', item)`, `('done', item)`, `('drop', item)` and return `(open items, done items)` in the order they were added. Adding an existing item does nothing.

**Examples**

- `commands = [('add', 'read spec'), ('add', 'write tests'), ('add', 'read spec'), ('done', 'read spec'), ('add', 'fix bug'), ('drop', 'write tests')]` → `(['fix bug'], ['read spec'])`

In [ ]:
def notes(commands):
    """
    Long tasks outlive the context window, so agents keep a to-do file. Apply
    commands ('add', item), ('done', item), ('drop', item) and return (open
    items, done items) in the order they were added. Adding an existing item
    does nothing.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(notes, [
    (([('add', 'read spec'), ('add', 'write tests'), ('add', 'read spec'), ('done', 'read spec'), ('add', 'fix bug'), ('drop', 'write tests')],), (['fix bug'], ['read spec'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two lists and if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One ordered dict of item → status.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two lists and if-statements.

```python
def notes(commands):
    open_items, done = [], []
    for cmd, item in commands:
        if cmd == "add" and item not in open_items and item not in done:
            open_items.append(item)
        elif cmd == "done" and item in open_items:
            open_items.remove(item)
            done.append(item)
        elif cmd == "drop" and item in open_items:
            open_items.remove(item)
    return (open_items, done)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One ordered dict of item → status.

```python
def notes(commands):
    status = {}
    for cmd, item in commands:
        if cmd == "add":
            status.setdefault(item, "open")
        elif status.get(item) == "open":
            status[item] = "done" if cmd == "done" else "dropped"
    order = list(status)
    return ([i for i in order if status[i] == "open"], [i for i in order if status[i] == "done"])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two lists and if-statements. That costs O(n²) time and O(n) space."

"I can do better: One ordered dict of item → status. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/a-notes-file-the-agent-keeps)

<a id="q7"></a>
### Q7. Newest Fact Wins

🟢 Warm-up · *Conflicting memories*

Memories arrive as `(key, value, timestamp)` in any order. Keep the value with the latest timestamp for each key. Return `{key: value}` sorted by key.

**Examples**

- `memories = [('city', 'Pune', 5), ('city', 'Mumbai', 9), ('diet', 'veg', 3), ('city', 'Delhi', 7)]` → `{'city': 'Mumbai', 'diet': 'veg'}`

In [ ]:
def newest(memories):
    """
    Memories arrive as (key, value, timestamp) in any order. Keep the value with
    the latest timestamp for each key. Return {key: value} sorted by key.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(newest, [
    (([('city', 'Pune', 5), ('city', 'Mumbai', 9), ('diet', 'veg', 3), ('city', 'Delhi', 7)],), {'city': 'Mumbai', 'diet': 'veg'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dict holding (timestamp, value) and comparisons.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort by timestamp, then a dict comprehension lets later ones overwrite.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A dict holding (timestamp, value) and comparisons.

```python
def newest(memories):
    best = {}
    for key, value, ts in memories:
        if key not in best or ts > best[key][0]:
            best[key] = (ts, value)
    return {k: best[k][1] for k in sorted(best)}
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sort by timestamp, then a dict comprehension lets later ones overwrite.

```python
def newest(memories):
    latest = {k: v for k, v, _ in sorted(memories, key=lambda m: m[2])}
    return dict(sorted(latest.items()))
```

**Cost:** O(n log n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dict holding (timestamp, value) and comparisons. That costs O(n) time and O(k) space."

"I can do better: Sort by timestamp, then a dict comprehension lets later ones overwrite. That's O(n log n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/newest-fact-wins)

<a id="q8"></a>
### Q8. What's Eating the Window?

🟢 Warm-up · *Context audits*

Given token counts per section, return `(sections sorted by share, largest first, as (name, % rounded to 1)), names taking over 40%)`. Auditing the context is the first step of any context-engineering fix.

**Examples**

- `sections = {'system': 1200, 'tools': 9000, 'history': 14000, 'docs': 6000, 'question': 80}` → `([('history', 46.2), ('tools', 29.7), ('docs', 19.8), ('system', 4.0), ('question', 0.3)], ['history'])`

In [ ]:
def window_audit(sections):
    """
    Given token counts per section, return (sections sorted by share, largest
    first, as (name, % rounded to 1)), names taking over 40%). Auditing the
    context is the first step of any context-engineering fix.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(window_audit, [
    (({'system': 1200, 'tools': 9000, 'history': 14000, 'docs': 6000, 'question': 80},), ([('history', 46.2), ('tools', 29.7), ('docs', 19.8), ('system', 4.0), ('question', 0.3)], ['history'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute shares in a loop, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sorted comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute shares in a loop, then sort.

```python
def window_audit(sections):
    total = 0
    for n in sections.values():
        total += n
    shares = []
    for name, n in sections.items():
        shares.append((name, round(n / total * 100, 1)))
    shares.sort(key=lambda x: -x[1])
    big = [name for name, pct in shares if pct > 40]
    return (shares, big)
```

**Cost:** O(s log s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sorted comprehension.

```python
def window_audit(sections):
    total = sum(sections.values())
    shares = sorted(((k, round(v / total * 100, 1)) for k, v in sections.items()), key=lambda x: -x[1])
    return (shares, [k for k, pct in shares if pct > 40])
```

**Cost:** O(s log s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute shares in a loop, then sort. That costs O(s log s) time and O(s) space."

"I can do better: A sorted comprehension. That's O(s log s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/whats-eating-the-window)

<a id="q9"></a>
### Q9. Fit the History Into the Window

🟡 Interview standard · *Trimming*

Always keep message 0 (the system prompt) and the newest message. Then add older messages from newest to oldest while the total stays ≤ `window`. Return the kept indexes in order.

**Examples**

- `sizes = [300, 900, 400, 700, 200, 100], window = 1300` → `[0, 3, 4, 5]`
- `sizes = [300, 900, 400, 700, 200, 100], window = 400` → `[0, 5]`

In [ ]:
def fit_window(sizes, window):
    """
    Always keep message 0 (the system prompt) and the newest message. Then add
    older messages from newest to oldest while the total stays ≤ window. Return
    the kept indexes in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fit_window, [
    (([300, 900, 400, 700, 200, 100], 1300), [0, 3, 4, 5]),
    (([300, 900, 400, 700, 200, 100], 400), [0, 5]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk backwards with a running total.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same walk, collecting into a set and sorting at the end.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk backwards with a running total.

```python
def fit_window(sizes, window):
    kept = [0, len(sizes) - 1]
    used = sizes[0] + sizes[-1]
    i = len(sizes) - 2
    while i > 0:
        if used + sizes[i] > window:
            break
        kept.append(i)
        used += sizes[i]
        i -= 1
    return sorted(kept)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same walk, collecting into a set and sorting at the end.

```python
def fit_window(sizes, window):
    kept, used = {0, len(sizes) - 1}, sizes[0] + sizes[-1]
    for i in range(len(sizes) - 2, 0, -1):
        if used + sizes[i] > window:
            break
        kept.add(i)
        used += sizes[i]
    return sorted(kept)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk backwards with a running total. That costs O(n) time and O(n) space."

"I can do better: Same walk, collecting into a set and sorting at the end. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/fit-the-history-into-the-window)

<a id="q10"></a>
### Q10. Compact the Conversation

🟡 Interview standard · *Summarise the oldest turns*

When the history (after the system prompt) exceeds `trigger` tokens, replace the oldest messages with one summary of `summary` tokens, keeping the newest `keep` messages. Return `(tokens before, tokens after, messages after including the system prompt)`.

**Examples**

- `sizes = [500, 1200, 3000, 2500, 1800, 900, 400], trigger = 6000, keep = 3, summary = 600` → `(10300, 4200, 5)`
- `sizes = [500, 100, 200], trigger = 6000, keep = 3, summary = 600` → `(800, 800, 3)`

In [ ]:
def compact(sizes, trigger, keep, summary):
    """
    When the history (after the system prompt) exceeds trigger tokens, replace
    the oldest messages with one summary of summary tokens, keeping the newest
    keep messages. Return (tokens before, tokens after, messages after including
    the system prompt).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compact, [
    (([500, 1200, 3000, 2500, 1800, 900, 400], 6000, 3, 600), (10300, 4200, 5)),
    (([500, 100, 200], 6000, 3, 600), (800, 800, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum, check, then rebuild the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slicing: system + [summary] + last keep.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sum, check, then rebuild the list.

```python
def compact(sizes, trigger, keep, summary):
    before = 0
    for s in sizes:
        before += s
    history = sizes[1:]
    if before - sizes[0] <= trigger:
        return (before, before, len(sizes))
    new = [sizes[0], summary]
    for s in history[len(history) - keep:]:
        new.append(s)
    return (before, sum(new), len(new))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slicing: system + [summary] + last keep.

```python
def compact(sizes, trigger, keep, summary):
    before = sum(sizes)
    if before - sizes[0] <= trigger:
        return (before, before, len(sizes))
    new = [sizes[0], summary, *sizes[-keep:]]
    return (before, sum(new), len(new))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum, check, then rebuild the list. That costs O(n) time and O(n) space."

"I can do better: Slicing: system + [summary] + last keep. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/compact-the-conversation)

<a id="q11"></a>
### Q11. Avoid 'Lost in the Middle'

🟡 Interview standard · *Ordering documents*

Models attend best to the start and end of a long context. Given document ids ranked by relevance (best first), place them so the best is first, the second-best is last, the third is second, the fourth second-to-last, and so on.

**Examples**

- `ranked = ['d1', 'd2', 'd3', 'd4', 'd5']` → `['d1', 'd3', 'd5', 'd4', 'd2']`
- `ranked = ['a', 'b']` → `['a', 'b']`

In [ ]:
def edges_first(ranked):
    """
    Models attend best to the start and end of a long context. Given document
    ids ranked by relevance (best first), place them so the best is first, the
    second-best is last, the third is second, the fourth second-to-last, and so
    on.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(edges_first, [
    ((['d1', 'd2', 'd3', 'd4', 'd5'],), ['d1', 'd3', 'd5', 'd4', 'd2']),
    ((['a', 'b'],), ['a', 'b']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fill a result list from both ends alternately.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Even ranks go to the front in order, odd ranks to the back reversed.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Fill a result list from both ends alternately.

```python
def edges_first(ranked):
    out = [None] * len(ranked)
    left, right = 0, len(ranked) - 1
    for i, d in enumerate(ranked):
        if i % 2 == 0:
            out[left] = d
            left += 1
        else:
            out[right] = d
            right -= 1
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Even ranks go to the front in order, odd ranks to the back reversed.

```python
def edges_first(ranked):
    return ranked[0::2] + ranked[1::2][::-1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fill a result list from both ends alternately. That costs O(n) time and O(n) space."

"I can do better: Even ranks go to the front in order, odd ranks to the back reversed. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/avoid-lost-in-the-middle)

<a id="q12"></a>
### Q12. What Does Caching Save?

🟡 Interview standard · *Prompt-cache pricing*

A request has `prefix` cacheable tokens and `rest` other input tokens, sent `n` times. Input costs `price` per million tokens; writing the cache costs 1.25× and reading it 0.1×. The first request writes the cache; later ones read it. Return `(cost without caching, cost with caching, % saved)`, money rounded to 4 and % to 1.

**Examples**

- `prefix = 50000, rest = 2000, n = 20, price = 3.0` → `(3.12, 0.5925, 81.0)`
- `prefix = 1000, rest = 2000, n = 2, price = 3.0` → `(0.018, 0.0161, 10.8)`

In [ ]:
def cache_savings(prefix, rest, n, price):
    """
    A request has prefix cacheable tokens and rest other input tokens, sent n
    times. Input costs price per million tokens; writing the cache costs 1.25×
    and reading it 0.1×. The first request writes the cache; later ones read it.
    Return (cost without caching, cost with caching, % saved), money rounded to
    4 and % to 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cache_savings, [
    ((50000, 2000, 20, 3.0), (3.12, 0.5925, 81.0)),
    ((1000, 2000, 2, 3.0), (0.018, 0.0161, 10.8)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add request by request.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A closed-form formula.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add request by request.

```python
def cache_savings(prefix, rest, n, price):
    per_token = price / 1_000_000
    plain = 0
    cached = 0
    for i in range(n):
        plain += (prefix + rest) * per_token
        if i == 0:
            cached += (prefix * 1.25 + rest) * per_token
        else:
            cached += (prefix * 0.1 + rest) * per_token
    return (round(plain, 4), round(cached, 4), round((plain - cached) / plain * 100, 1))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A closed-form formula.

```python
def cache_savings(prefix, rest, n, price):
    p = price / 1e6
    plain = n * (prefix + rest) * p
    cached = (prefix * 1.25 + (n - 1) * prefix * 0.1 + n * rest) * p
    return (round(plain, 4), round(cached, 4), round((plain - cached) / plain * 100, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add request by request. That costs O(n) time and O(1) space."

"I can do better: A closed-form formula. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/what-does-caching-save)

<a id="q13"></a>
### Q13. Share the Budget Between Sections

🟡 Interview standard · *Allocation*

Split `budget` tokens between sections. Each gets its `minimum` first; the remainder is shared in proportion to `weight` (rounded down). No section gets more than it `wants`. Return `{section: tokens}`.

**Examples**

- `sections = {'system': (500, 0, 500), 'memory': (200, 1, 3000), 'docs': (1000, 3, 20000), 'history': (500, 2, 6000)}, budget = 8000` → `{'system': 500, 'memory': 1166, 'docs': 3900, 'history': 2433}`

In [ ]:
def allocate(sections, budget):
    """
    Split budget tokens between sections. Each gets its minimum first; the
    remainder is shared in proportion to weight (rounded down). No section gets
    more than it wants. Return {section: tokens}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(allocate, [
    (({'system': (500, 0, 500), 'memory': (200, 1, 3000), 'docs': (1000, 3, 20000), 'history': (500, 2, 6000)}, 8000), {'system': 500, 'memory': 1166, 'docs': 3900, 'history': 2433}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Give minimums, then loop over sections computing shares.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same steps with comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Give minimums, then loop over sections computing shares.

```python
def allocate(sections, budget):
    out = {}
    left = budget
    for name, (minimum, weight, wants) in sections.items():
        out[name] = minimum
        left -= minimum
    total_w = 0
    for name, (minimum, weight, wants) in sections.items():
        total_w += weight
    for name, (minimum, weight, wants) in sections.items():
        extra = left * weight // total_w if total_w else 0
        out[name] = min(wants, minimum + extra)
    return out
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same steps with comprehensions.

```python
def allocate(sections, budget):
    left = budget - sum(m for m, _, _ in sections.values())
    total_w = sum(w for _, w, _ in sections.values()) or 1
    return {name: min(wants, m + left * w // total_w) for name, (m, w, wants) in sections.items()}
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Give minimums, then loop over sections computing shares. That costs O(s) time and O(s) space."

"I can do better: Same steps with comprehensions. That's O(s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/share-the-budget-between-sections)

<a id="q14"></a>
### Q14. Choose Few-Shot Examples

🟡 Interview standard · *Dynamic examples*

Pick the `k` examples whose words overlap most with the query (Jaccard similarity of lower-cased word sets); ties keep the original order.

**Examples**

- `query = 'refund my late order', examples = ['where is my order', 'refund for a damaged item', 'change my password', 'late delivery refund please'], k = 2` → `['where is my order', 'late delivery refund please']`

In [ ]:
def pick_examples(query, examples, k):
    """
    Pick the k examples whose words overlap most with the query (Jaccard
    similarity of lower-cased word sets); ties keep the original order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pick_examples, [
    (('refund my late order', ['where is my order', 'refund for a damaged item', 'change my password', 'late delivery refund please'], 2), ['where is my order', 'late delivery refund please']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Score each example with set operations, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted` with a key, which is stable for ties.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Score each example with set operations, then sort.

```python
def pick_examples(query, examples, k):
    q = set(query.lower().split())
    scored = []
    for i, ex in enumerate(examples):
        e = set(ex.lower().split())
        scored.append((len(q & e) / len(q | e), i, ex))
    scored.sort(key=lambda t: (-t[0], t[1]))
    return [ex for _, _, ex in scored[:k]]
```

**Cost:** O(e · w) time, O(e) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `sorted` with a key, which is stable for ties.

```python
def pick_examples(query, examples, k):
    q = set(query.lower().split())
    jac = lambda ex: len(q & set(ex.lower().split())) / len(q | set(ex.lower().split()))
    return sorted(examples, key=jac, reverse=True)[:k]
```

**Cost:** O(e · w) time, O(e) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Score each example with set operations, then sort. That costs O(e · w) time and O(e) space."

"I can do better: `sorted` with a key, which is stable for ties. That's O(e · w) time and O(e) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/choose-few-shot-examples)

<a id="q15"></a>
### Q15. The Hidden Cost of Long Chats

🟡 Interview standard · *History is re-sent every turn*

Every turn re-sends the whole history. Each turn adds `per_turn` tokens. Return the total input tokens over `turns` turns **(a)** without compaction and **(b)** when history is compacted to `summary` tokens whenever it would exceed `trigger`.

**Examples**

- `turns = 40, per_turn = 2000, trigger = 30000, summary = 3000` → `(1640000, 653000)`

In [ ]:
def chat_cost(turns, per_turn, trigger, summary):
    """
    Every turn re-sends the whole history. Each turn adds per_turn tokens.
    Return the total input tokens over turns turns (a) without compaction and
    (b) when history is compacted to summary tokens whenever it would exceed
    trigger.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chat_cost, [
    ((40, 2000, 30000, 3000), (1640000, 653000)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate turn by turn for both strategies.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Closed form for (a): per\_turn × t(t+1)/2; simulate (b).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate turn by turn for both strategies.

```python
def chat_cost(turns, per_turn, trigger, summary):
    plain = 0
    hist = 0
    for _ in range(turns):
        hist += per_turn
        plain += hist
    compacted = 0
    hist = 0
    for _ in range(turns):
        hist += per_turn
        if hist > trigger:
            hist = summary + per_turn
        compacted += hist
    return (plain, compacted)
```

**Cost:** O(t) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Closed form for (a): per\_turn × t(t+1)/2; simulate (b).

```python
def chat_cost(turns, per_turn, trigger, summary):
    plain = per_turn * turns * (turns + 1) // 2
    hist = compacted = 0
    for _ in range(turns):
        hist = hist + per_turn if hist + per_turn <= trigger else summary + per_turn
        compacted += hist
    return (plain, compacted)
```

**Cost:** O(t) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate turn by turn for both strategies. That costs O(t) time and O(1) space."

"I can do better: Closed form for (a): per\_turn × t(t+1)/2; simulate (b). That's O(t) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/the-hidden-cost-of-long-chats)

<a id="q16"></a>
### Q16. Extract Facts to Remember

🟡 Interview standard · *Memory extraction*

Pull simple facts from a user's messages with `PATTERNS` (case-insensitive). Later messages override earlier ones. Return `{key: value}`. (Real systems ask an LLM to extract facts; the pipeline around it is the same.)

**Examples**

- `messages = ['Hi, my name is Asha', 'I live in Pune and I prefer email', 'Actually I live in Mumbai now']` → `{'name': 'Asha', 'city': 'Mumbai', 'preference': 'email'}`

In [ ]:
import re

PATTERNS = [(r"my name is (\w+)", "name"), (r"i live in (\w+)", "city"), (r"i prefer (\w+)", "preference"),
            (r"i work (?:at|for) (\w+)", "employer")]

def extract_facts(messages):
    """
    Pull simple facts from a user's messages with PATTERNS (case-insensitive).
    Later messages override earlier ones. Return {key: value}. (Real systems ask
    an LLM to extract facts; the pipeline around it is the same.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(extract_facts, [
    ((['Hi, my name is Asha', 'I live in Pune and I prefer email', 'Actually I live in Mumbai now'],), {'name': 'Asha', 'city': 'Mumbai', 'preference': 'email'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over messages and patterns with `re.search`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict comprehension over every match, in order — later keys overwrite earlier ones.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over messages and patterns with `re.search`.

```python
def extract_facts(messages):
    facts = {}
    for m in messages:
        for pattern, key in PATTERNS:
            found = re.search(pattern, m, re.I)
            if found:
                facts[key] = found.group(1)
    return facts
```

**Cost:** O(m · p) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict comprehension over every match, in order — later keys overwrite earlier ones.

```python
def extract_facts(messages):
    return {key: m.group(1) for msg in messages for pattern, key in PATTERNS if (m := re.search(pattern, msg, re.I))}
```

**Cost:** O(m · p) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over messages and patterns with `re.search`. That costs O(m · p) time and O(k) space."

"I can do better: A dict comprehension over every match, in order — later keys overwrite earlier ones. That's O(m · p) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/extract-facts-to-remember)

<a id="q17"></a>
### Q17. Add, Update or Skip?

🟡 Interview standard · *Memory consolidation*

For each new fact `(key, value)` decide against the store: 'ADD' if the key is new, 'SKIP' if the value is the same (ignoring case), 'UPDATE' otherwise. Apply them in order, keeping replaced values in a history list. Return `(decisions, final store, history)`.

**Examples**

- `store = {'city': 'Pune', 'name': 'Asha'}, facts = [('city', 'pune'), ('city', 'Mumbai'), ('diet', 'vegetarian'), ('name', 'Asha')]` → `(['SKIP', 'UPDATE', 'ADD', 'SKIP'], {'city': 'Mumbai', 'name': 'Asha', 'diet': 'vegetarian'}, [('city', 'Pune')])`

In [ ]:
def consolidate(store, facts):
    """
    For each new fact (key, value) decide against the store: 'ADD' if the key is
    new, 'SKIP' if the value is the same (ignoring case), 'UPDATE' otherwise.
    Apply them in order, keeping replaced values in a history list. Return
    (decisions, final store, history).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(consolidate, [
    (({'city': 'Pune', 'name': 'Asha'}, [('city', 'pune'), ('city', 'Mumbai'), ('diet', 'vegetarian'), ('name', 'Asha')]), (['SKIP', 'UPDATE', 'ADD', 'SKIP'], {'city': 'Mumbai', 'name': 'Asha', 'diet': 'vegetarian'}, [('city', 'Pune')])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain that also updates the store.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same logic with a helper returning the decision.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if-chain that also updates the store.

```python
def consolidate(store, facts):
    store = dict(store)
    decisions, history = [], []
    for key, value in facts:
        if key not in store:
            decisions.append("ADD")
            store[key] = value
        elif store[key].lower() == value.lower():
            decisions.append("SKIP")
        else:
            decisions.append("UPDATE")
            history.append((key, store[key]))
            store[key] = value
    return (decisions, store, history)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same logic with a helper returning the decision.

```python
def consolidate(store, facts):
    store, decisions, history = dict(store), [], []
    for key, value in facts:
        old = store.get(key)
        op = "ADD" if old is None else "SKIP" if old.lower() == value.lower() else "UPDATE"
        decisions.append(op)
        if op == "UPDATE":
            history.append((key, old))
        if op != "SKIP":
            store[key] = value
    return (decisions, store, history)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain that also updates the store. That costs O(n) time and O(n) space."

"I can do better: Same logic with a helper returning the decision. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/add-update-or-skip)

<a id="q18"></a>
### Q18. Recall the Right Memories

🟡 Interview standard · *Relevance × recency*

Score each memory `(text, age_days)` as (number of query words it contains) × 0.5^(age / half\_life). Return the top `k` texts with a score above 0 (ties: newer first).

**Examples**

- `query = 'how should I contact you about my refund', half_life = 30, k = 2` → `['asked about refund for order 812']`
- `query = 'is the user allergic to anything', half_life = 30, k = 3` → `['allergic to peanuts']`

In [ ]:
MEMORIES = [("prefers email over phone", 30), ("lives in Mumbai", 2), ("allergic to peanuts", 200),
            ("asked about refund for order 812", 1), ("prefers window seats on flights", 90)]

def recall(query, half_life, k):
    """
    Score each memory (text, age_days) as (number of query words it contains) ×
    0.5^(age / half_life). Return the top k texts with a score above 0 (ties:
    newer first).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(recall, [
    (('how should I contact you about my refund', 30, 2), ['asked about refund for order 812']),
    (('is the user allergic to anything', 30, 3), ['allergic to peanuts']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Score in a loop and sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A key function with `sorted`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Score in a loop and sort.

```python
def recall(query, half_life, k):
    q = set(query.lower().split())
    scored = []
    for text, age in MEMORIES:
        overlap = 0
        for w in text.lower().split():
            if w in q:
                overlap += 1
        score = overlap * 0.5 ** (age / half_life)
        if score > 0:
            scored.append((score, -age, text))
    scored.sort(reverse=True)
    return [t for _, _, t in scored[:k]]
```

**Cost:** O(m · w) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A key function with `sorted`.

```python
def recall(query, half_life, k):
    q = set(query.lower().split())
    score = lambda m: sum(w in q for w in m[0].lower().split()) * 0.5 ** (m[1] / half_life)
    ranked = sorted((m for m in MEMORIES if score(m) > 0), key=lambda m: (score(m), -m[1]), reverse=True)
    return [t for t, _ in ranked[:k]]
```

**Cost:** O(m · w) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Score in a loop and sort. That costs O(m · w) time and O(m) space."

"I can do better: A key function with `sorted`. That's O(m · w) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/recall-the-right-memories)

<a id="q19"></a>
### Q19. Assemble the Final Prompt

🟡 Interview standard · *Putting it together*

Build the context: system prompt, then recalled memories as a bullet list under 'Known about the user:', then documents in 'lost in the middle' order, then the question. Skip any section that is empty. Sections are separated by a blank line. Return the text.

**Examples**

- `system = "You are ShopKart's assistant.", memories = ['lives in Mumbai'], docs = ['d1', 'd2', 'd3'], question = 'Where is my order?'` → `"You are ShopKart's assistant.\n\nKnown about the user:\n- lives in Mumbai\n\nd1\nd3\nd2\n\nWhere is my order?"`
- `system = 'Be brief.', memories = [], docs = [], question = 'Hi'` → `'Be brief.\n\nHi'`

In [ ]:
def assemble(system, memories, docs, question):
    """
    Build the context: system prompt, then recalled memories as a bullet list
    under 'Known about the user:', then documents in 'lost in the middle' order,
    then the question. Skip any section that is empty. Sections are separated by
    a blank line. Return the text.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(assemble, [
    (("You are ShopKart's assistant.", ['lives in Mumbai'], ['d1', 'd2', 'd3'], 'Where is my order?'), "You are ShopKart's assistant.\n\nKnown about the user:\n- lives in Mumbai\n\nd1\nd3\nd2\n\nWhere is my order?"),
    (('Be brief.', [], [], 'Hi'), 'Be brief.\n\nHi'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Append sections to a list with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A list of optional sections filtered for empties, joined once.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Append sections to a list with if-statements.

```python
def assemble(system, memories, docs, question):
    parts = [system]
    if memories:
        text = "Known about the user:"
        for m in memories:
            text += "\n- " + m
        parts.append(text)
    if docs:
        ordered = [None] * len(docs)
        left, right = 0, len(docs) - 1
        for i, d in enumerate(docs):
            if i % 2 == 0:
                ordered[left] = d
                left += 1
            else:
                ordered[right] = d
                right -= 1
        parts.append("\n".join(ordered))
    parts.append(question)
    return "\n\n".join(parts)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A list of optional sections filtered for empties, joined once.

```python
def assemble(system, memories, docs, question):
    sections = [system,
                "Known about the user:\n" + "\n".join(f"- {m}" for m in memories) if memories else "",
                "\n".join(docs[0::2] + docs[1::2][::-1]),
                question]
    return "\n\n".join(s for s in sections if s)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Append sections to a list with if-statements. That costs O(n) time and O(n) space."

"I can do better: A list of optional sections filtered for empties, joined once. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/assemble-the-final-prompt)

<a id="q20"></a>
### Q20. Is the Prefix Cached?

🟡 Interview standard · *Cache hits*

A cache hit needs the new request to start with exactly the same blocks as an earlier one. Count the tokens of the leading blocks that match (blocks are `(text, tokens)`). It only counts as a hit if those tokens reach `minimum`. Return `(matching prefix tokens, hit?)`.

**Examples**

- `old = [('SYS', 900), ('TOOLS', 1500), ('DOCS', 4000), ('Q1', 50)], new = [('SYS', 900), ('TOOLS', 1500), ('DOCS', 4000), ('Q2', 60)], minimum = 1024` → `(6400, True)`
- `old = [('SYS', 900), ('TOOLS', 1500)], new = [('SYS v2', 905), ('TOOLS', 1500)], minimum = 1024` → `(0, False)`
- `old = [('SYS', 500), ('Q1', 20)], new = [('SYS', 500), ('Q2', 20)], minimum = 1024` → `(500, False)`

In [ ]:
def prefix_hit(old, new, minimum):
    """
    A cache hit needs the new request to start with exactly the same blocks as
    an earlier one. Count the tokens of the leading blocks that match (blocks
    are (text, tokens)). It only counts as a hit if those tokens reach minimum.
    Return (matching prefix tokens, hit?).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(prefix_hit, [
    (([('SYS', 900), ('TOOLS', 1500), ('DOCS', 4000), ('Q1', 50)], [('SYS', 900), ('TOOLS', 1500), ('DOCS', 4000), ('Q2', 60)], 1024), (6400, True)),
    (([('SYS', 900), ('TOOLS', 1500)], [('SYS v2', 905), ('TOOLS', 1500)], 1024), (0, False)),
    (([('SYS', 500), ('Q1', 20)], [('SYS', 500), ('Q2', 20)], 1024), (500, False)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk both lists until the first difference.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.takewhile` over the zipped blocks.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk both lists until the first difference.

```python
def prefix_hit(old, new, minimum):
    total = 0
    for a, b in zip(old, new):
        if a != b:
            break
        total += a[1]
    return (total, total >= minimum)
```

**Cost:** O(b) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.takewhile` over the zipped blocks.

```python
def prefix_hit(old, new, minimum):
    from itertools import takewhile
    total = sum(a[1] for a, b in takewhile(lambda p: p[0] == p[1], zip(old, new)))
    return (total, total >= minimum)
```

**Cost:** O(b) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk both lists until the first difference. That costs O(b) time and O(1) space."

"I can do better: `itertools.takewhile` over the zipped blocks. That's O(b) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/context-memory/is-the-prefix-cached)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why not just use the biggest context window?</b></summary>

Cost and latency grow with input size, and models use information less reliably as the context fills, especially in the middle. Curated, smaller context usually gives better answers.

</details>

<details>
<summary><b>2. How does prompt caching work and how do you get hits?</b></summary>

The provider stores the processed prefix of a recent prompt and reuses it if a new prompt starts with exactly the same content. Put stable content first and the changing parts last, and keep the prefix byte-identical.

</details>

<details>
<summary><b>3. Trimming vs compaction?</b></summary>

Trimming drops old turns and loses their information. Compaction replaces them with a summary, keeping decisions and facts at a fraction of the tokens.

</details>

<details>
<summary><b>4. How would you build long-term memory for an assistant?</b></summary>

Extract facts after each conversation, consolidate them (add, update with history, skip duplicates), store them with timestamps, and recall the most relevant and recent few into the prompt. Let users view and delete them.

</details>

<details>
<summary><b>5. What is 'lost in the middle'?</b></summary>

Models tend to use information at the start and end of a long context better than in the middle, so put the most important material at the edges.

</details>

<details>
<summary><b>6. What is just-in-time context?</b></summary>

Giving the agent references (file names, IDs) and tools to fetch details when needed, instead of loading everything up front.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: From prompt engineering to context engineering
- [ ] I can explain: Keeping long conversations inside the window
- [ ] I can explain: Order matters: caching and 'lost in the middle'
- [ ] I can explain: Choosing what to include
- [ ] I can explain: Memory that lasts beyond one conversation

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Extract Facts to Remember
- [ ] [Q17](#q17) Add, Update or Skip?
- [ ] [Q18](#q18) Recall the Right Memories
- [ ] [Q19](#q19) Assemble the Final Prompt
- [ ] [Q20](#q20) Is the Prefix Cached?

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.